In [1]:
from src.retrievers.hybrid import HybridRetriever
from src.data_utils import load_documents, chunker
from src.generate import generate

documents = load_documents("/home/nick/new-speedrun/data/2038.pdf")
chunks = chunker(documents, chunk_size=301, chunk_overlap=30)

hybrid = HybridRetriever()
hybrid.add_documents(chunks)


/home/nick/new-speedrun/venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10496.21it/s]


In [2]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

query = 'What did Darren Walsh do for Work in 2027?'
retrieved = hybrid.retrieve(query, top_k=20)
pairs = [(query, doc.page_content) for doc in retrieved]
scores = reranker.predict(pairs)


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 8072.47it/s]


In [3]:
ranked = sorted(zip(scores, retrieved), key=lambda item: item[0], reverse=True)


In [4]:

test_queries = [
    'Does Sterling Bronson have a mag?',
    'What did Darren Walsh do for Work in 2027?',
    'Which pub does Winston Reed regularly visit?',
    'What town is this novel set in?',
    'Who is Grace?',
    'Which chapter first introduces Cassie May?',
    'Which social or occupational class does Cassie belong to?',
    'What are Cats?',
    'what are transistors?',
    'does sterling own the transistors?',
    'what are the people who own the transistors referred to?'
]


def rag_answer(query: str, k: int = 40) -> str:
    if k < 1:
        raise ValueError("k must be at least 1")
    retrieved = hybrid.retrieve(query, top_k=40)
    if not retrieved:
        return "I do not know"
    pairs = [(query, doc.page_content) for doc in retrieved]
    scores = reranker.predict(pairs)
    ranked = sorted(zip(scores, retrieved), key=lambda item: item[0], reverse=True)
    context_chunks = [doc.page_content for _, doc in ranked[:k]]
    return generate(query, context_chunks)



for query in test_queries:
    print(query)
    print(rag_answer(query))
    print()



Does Sterling Bronson have a mag?
Yes, Sterling Bronson has a Mag. The Clerk is the only Mag at Ashcombe, and Sterling made it himself.

What did Darren Walsh do for Work in 2027?
Darren Walsh was a scaffolder in 2027.

Which pub does Winston Reed regularly visit?
The Crown

What town is this novel set in?
The novel is set in the town of Brackton.

Who is Grace?
Grace is a character in the novel "2038." She appears to be a significant figure in Reed's life, someone who knows him well and shares a close, intimate relationship with him. Grace is depicted as being perceptive, understanding, and supportive, often knowing what Reed needs or feels without him having to express it. She is described as being on the sofa with her feet tucked under her, looking at Reed as though he were the best thing that had happened all day. Grace seems to have a calming and reassuring presence in Reed's life.

Which chapter first introduces Cassie May?
Cassie May is first introduced in Chapter Three.

Which 